In [16]:
import torch
import collections # collections 모듈에 있는 deque를 사용하기 위함
#dequq는 double-ended queue로써, queue와 stack의 기능을 다 쓸 수 있다.

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import numpy as np
import copy


In [17]:
class enva():
    def __init__(self):
        self.height = 10
        self.width = 10
        self.num_obstacle_range =5
        self.num_obstacle_range_min = 3
        self.turn = 0
        self.position = int(round(self.width/2))
        self.map = np.array([[0.0 for j in range(self.width)] for i in range(self.height)])
        self.map[self.height-1][self.position] = 2.
        self.done = False
        
    def step(self, key):
        #obstacle down
        i = self.height - 2
        while(i>=0):
            self.map[i+1] = self.map[i]
            i -= 1
        #obstacle init(first_line)
        self.map[0] = np.array([0 for j in range(self.width)])
        #obstacle making
        num_obstacle = random.randrange(self.num_obstacle_range_min,self.num_obstacle_range)
        i = 0
        while(i < num_obstacle):
            position_obstacle = random.randrange(0,self.width)
            if (self.map[0][position_obstacle] == 1.0):
                continue
            self.map[0][position_obstacle] = 1.0
            i += 1
        #big_obstacle
        big_obstacle = random.randrange(0,20)
        if(big_obstacle == -1): ############### 1로 바꾸면 큰 똥이 생긴다.
            big_obstacle = random.randrange(1,self.width-1)
            for j in range(3):
                for k in range(-1,2):
                    self.map[j][big_obstacle + k] = 1.
        #player position
        reward = 1
        # 0 : 왼쪽, 1 : 가만히, 2 : 오른쪽
        if key == 0:
            if(self.position>0):
                self.position -= 1
            
        if key == 1:
            pass      
        if key == 2:
            if(self.position<self.width-1):
                self.position += 1
                  
        if (self.map[self.height-1][self.position] == 1.0):
                    reward = 0
                    self.done = True
                    #print("====Game Over====")
                        
        self.map[self.height-1][self.position] = 2.0
        #reward(turn(time))
        self.turn += 1
        
        return torch.flatten(torch.tensor(self.map),0), reward, self.done, _ 

    def reset(self):
        self.turn = 0
        self.position = round(self.width/2)
        self.map = np.array([[0.0 for j in range(self.width)] for i in range(self.height)])
        self.map[self.height-1][self.position] = 2.0
        self.done = False
        
        return torch.flatten(torch.tensor(self.map),0)

In [18]:
#nerual network setting -> good performance
#hyperparameters
buffer_size =30000   # environment's size에 맞춰서 setting
batch_size = 32   
gamma = 0.98  # Large future weight

class Agent(nn.Module):
    def __init__(self):
        super(Agent,self).__init__()
        self.fc1 = nn.Linear(100,128)
        self.fc2 = nn.Linear(128,128)
        self.fc3 = nn.Linear(128,128)
        self.fc4 = nn.Linear(128,3)  # 좌우, 제자리
  
    def forward(self,x):
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x
  
  
    def select_action(self,state,eps):   # state, reward를 받으면 action이 나오는 relation. 
                                        # but reward는 memory를 통해 사용하므로 state를 받아서 action을 뽑는다.

        coin = random.random() # 0~1
        out_value = self.forward(state) # 
        if coin < eps:   # random action
            a = random.randint(0,2) # 0 ,1 ,2 choice
        else : # optimize action
            a = torch.argmax(out_value).item() # a는 tensor가 아닌 int로 들어가야 한다.
                                               # 둘 중 높은 value를 가진 index가 action이다. 
        return a

class replay_buffer():
    def __init__(self):
        self.buffer = collections.deque(maxlen = buffer_size)
    def put(self,input): # buffer에 store
        self.buffer.append(input) # dtype = tensor  4개가 한 묶음으로 들어가야됨

    def sampling(self,bat_ch): 
        mini_batch = random.sample(self.buffer,bat_ch) # bat_ch X 4 (list), dtype = tensor
                                                       # buffer에서 batch_size (현재 32)만큼 sampling을 한다.
        #s_list, a_list, r_prime_list, s_prime_list,done_mask_lst = [],[],[],[],[] # list로 각각을 저장할거임
        a_list, r_prime_list, done_mask_lst = [],[],[]
        s_list = torch.tensor([0 for i in range(100)],dtype = torch.float).unsqueeze(dim=0)
        s_prime_list = torch.tensor([0 for i in range(100)],dtype = torch.float).unsqueeze(dim=0)
        
        for transition in mini_batch:
              s, a, r_prime, s_prime,done_mask = transition
              
              
              # a : int, r_prime : float
              s_list = torch.cat([s_list,s.unsqueeze(dim=0)],dim=0) 
              s_prime_list = torch.cat([s_prime_list,s_prime.unsqueeze(dim=0)],dim=0)         
              #s_list.append(s)
              a_list.append([a]) # 열로 쌓는다
              r_prime_list.append([r_prime])
              #s_prime_list.append(s_prime)
              done_mask_lst.append([done_mask])
        s_list = s_list[1:,:]
        s_prime_list = s_prime_list[1:,:]      
        return torch.tensor(s_list, dtype=torch.float), torch.tensor(a_list), \
                       torch.tensor(r_prime_list), torch.tensor(s_prime_list, dtype=torch.float),torch.tensor(done_mask_lst)

    def size(self):
        return len(self.buffer)

def train(q,q_target,disk,optimizer):
    for i in range(10):
        s, a, r, s_prime,done_mask = disk.sampling(batch_size) # s : batch_size x 4    

        q_out = q(s) # current
        next_q_out = q(s_prime)
        next_q_tar_out = q_target(s_prime)
        q_a = q_out.gather(1,a)

        #print(next_q_out.size()) 32 by 2 
        #print(next_q_out.max(1)[0]) max value, max index가 나와서 [0]을 취했는데 DDQN에선 Qseta의 argmax를 뽑아야 하므로 [1]
        max_q_prime = next_q_tar_out.gather(1,next_q_out.max(1)[1].unsqueeze(1))
        #max_q_prime = q_target(s_prime).max(1)[0].unsqueeze(1) 기존의 DQN
        target = r + gamma * max_q_prime * done_mask
        
        loss = F.smooth_l1_loss(q_a,target.detach())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [21]:
env = enva()
q = Agent()
q_target = Agent()
q_target.load_state_dict(q.state_dict())
memory = replay_buffer()  

optimizer = optim.Adam(q.parameters(),lr = 0.0005)
score = 0.0
best_score = 1500#0.
q.load_state_dict(torch.load('DQN_ddong_with_dongmin.pt'))
for epi in range(100000):

    eps = max(0.002, 0.08 - 0.01*(epi/200)) #0,08
    s = env.reset()
   
    done = False
    one_score = 0
    while not done :
   
        a = q.select_action(s.float(),eps)
        s_prime,r,done,info = env.step(a)
        done_mask = 0.0 if done else 1.0  # termination이면 done을 0으로..
        memory.put((s,a,r,s_prime, done_mask)) # state, action, reward, state', done_mask
        s = copy.deepcopy(s_prime)
        
        score += r
        one_score += r
    if one_score > best_score:
        best_score = one_score
        print("...save model...")
        torch.save(q.state_dict(),"DQN_ddong_with_dongmin.pt")
        
        print("best score : ",best_score)
    if memory.size() >3000 :
        train(q,q_target,memory,optimizer)

    if epi%100==0 and epi!=0:
        q_target.load_state_dict(q.state_dict())
        print("n_episode :{}, score : {:.1f}, n_buffer : {},eps = {}".format(
                                                        epi, score/100, memory.size(),eps))
        
        score = 0.0





...save model...
best score :  10
...save model...
best score :  13
...save model...
best score :  19
n_episode :100, score : 11.2, n_buffer : 1223,eps = 0.075
...save model...
best score :  24
...save model...
best score :  26
n_episode :200, score : 12.5, n_buffer : 2573,eps = 0.07


/tmp/ipykernel_1692662/3710646988.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(s_list, dtype=torch.float), torch.tensor(a_list), \
/tmp/ipykernel_1692662/3710646988.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(r_prime_list), torch.tensor(s_prime_list, dtype=torch.float),torch.tensor(done_mask_lst)


n_episode :300, score : 11.5, n_buffer : 3823,eps = 0.065
n_episode :400, score : 12.6, n_buffer : 5181,eps = 0.06
...save model...
best score :  27
n_episode :500, score : 12.6, n_buffer : 6541,eps = 0.055
n_episode :600, score : 12.6, n_buffer : 7901,eps = 0.05
n_episode :700, score : 13.2, n_buffer : 9317,eps = 0.045
...save model...
best score :  29
...save model...
best score :  38
n_episode :800, score : 13.2, n_buffer : 10741,eps = 0.04
n_episode :900, score : 13.5, n_buffer : 12190,eps = 0.035
n_episode :1000, score : 14.5, n_buffer : 13736,eps = 0.03
n_episode :1100, score : 14.4, n_buffer : 15281,eps = 0.025
n_episode :1200, score : 15.1, n_buffer : 16891,eps = 0.020000000000000004
n_episode :1300, score : 14.7, n_buffer : 18464,eps = 0.015
...save model...
best score :  48
n_episode :1400, score : 16.2, n_buffer : 20188,eps = 0.009999999999999995
n_episode :1500, score : 16.7, n_buffer : 21954,eps = 0.0050000000000000044
n_episode :1600, score : 17.0, n_buffer : 23757,eps = 

n_episode :12500, score : 100.0, n_buffer : 30000,eps = 0.002
...save model...
best score :  817
n_episode :12600, score : 136.2, n_buffer : 30000,eps = 0.002
n_episode :12700, score : 106.5, n_buffer : 30000,eps = 0.002
n_episode :12800, score : 94.6, n_buffer : 30000,eps = 0.002
n_episode :12900, score : 79.0, n_buffer : 30000,eps = 0.002
n_episode :13000, score : 104.4, n_buffer : 30000,eps = 0.002
n_episode :13100, score : 100.3, n_buffer : 30000,eps = 0.002
n_episode :13200, score : 85.0, n_buffer : 30000,eps = 0.002
n_episode :13300, score : 91.7, n_buffer : 30000,eps = 0.002
n_episode :13400, score : 96.4, n_buffer : 30000,eps = 0.002
n_episode :13500, score : 102.7, n_buffer : 30000,eps = 0.002
n_episode :13600, score : 94.0, n_buffer : 30000,eps = 0.002
n_episode :13700, score : 95.4, n_buffer : 30000,eps = 0.002
n_episode :13800, score : 105.6, n_buffer : 30000,eps = 0.002
n_episode :13900, score : 83.1, n_buffer : 30000,eps = 0.002
n_episode :14000, score : 108.5, n_buffer :

n_episode :25700, score : 118.7, n_buffer : 30000,eps = 0.002
n_episode :25800, score : 131.8, n_buffer : 30000,eps = 0.002
n_episode :25900, score : 87.5, n_buffer : 30000,eps = 0.002
n_episode :26000, score : 102.5, n_buffer : 30000,eps = 0.002
n_episode :26100, score : 85.0, n_buffer : 30000,eps = 0.002
n_episode :26200, score : 110.5, n_buffer : 30000,eps = 0.002
n_episode :26300, score : 94.8, n_buffer : 30000,eps = 0.002
n_episode :26400, score : 106.5, n_buffer : 30000,eps = 0.002
n_episode :26500, score : 132.8, n_buffer : 30000,eps = 0.002
n_episode :26600, score : 112.9, n_buffer : 30000,eps = 0.002
n_episode :26700, score : 109.7, n_buffer : 30000,eps = 0.002
n_episode :26800, score : 100.5, n_buffer : 30000,eps = 0.002
n_episode :26900, score : 112.7, n_buffer : 30000,eps = 0.002
n_episode :27000, score : 101.4, n_buffer : 30000,eps = 0.002
n_episode :27100, score : 85.2, n_buffer : 30000,eps = 0.002
n_episode :27200, score : 111.6, n_buffer : 30000,eps = 0.002
n_episode :2

KeyboardInterrupt: 